# Streaming

**Prerequisites:** `01_your_first_llm.ipynb`, `06_function_calling.ipynb`

Every notebook so far waited for the full reply before printing anything. `stream_chat()` (built back in notebook 1) already streams — this notebook is where that actually gets explained: what a streamed chunk looks like, why `finish_reason` matters, why it matters for latency, and what changes when tool calls are involved.

**By the end you'll have:**
- Proven a reply arrives in separate pieces — as static, saved-output proof, and as an artificially-slowed demo you can actually watch happen live (real Groq speed turns out to be too fast for the eye to perceive as typing at all)
- Seen `finish_reason` take three different real values (`stop`, `length`, `tool_calls`) and why code has to branch on each differently
- Found a Groq-specific detail: `usage` is included on the final chunk automatically
- Measured the real latency difference streaming makes
- Checked whether tool call arguments arrive fragmented on this stack (they don't — but seen why you can't assume that everywhere)
- Built `stream_and_print()` in `llm.py`

In [2]:
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from llm import client, stream_chat, DEFAULT_MODEL

## Streaming tokens

`stream=True` turns one response into many small pieces you iterate over, instead of one object you wait for. A printed transcript can't show an animation, so let's prove the pieces are real and separate the honest way — number each one as it arrives:

In [3]:
stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
)

for i, chunk in enumerate(stream):
    delta = chunk.choices[0].delta.content
    if delta:
        print(f"chunk {i:>2}: {delta!r}")

chunk  1: '1'
chunk  2: ','
chunk  3: ' '
chunk  4: '2'
chunk  5: ','
chunk  6: ' '
chunk  7: '3'
chunk  8: ','
chunk  9: ' '
chunk 10: '4'
chunk 11: ','
chunk 12: ' '
chunk 13: '5'
chunk 14: '.'


14 separate lines for one short sentence — that list *is* the proof. This is exactly what a chat UI receives too: not one string, but a sequence of tiny pieces arriving over time, and it's up to the receiver to do something with each one as it shows up.

A real UI doesn't print each piece on its own line, though — it prints with no separator, so the pieces visually merge into normal-looking text:

In [4]:
stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
)

full_reply = ""
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        full_reply += delta

print()
print()
print("accumulated:", repr(full_reply))

1, 2, 3, 4, 5.

accumulated: '1, 2, 3, 4, 5.'


Two separate reasons this might not have *looked* like typing when you ran it:

1. **A saved notebook can't show an animation at all** — it just stores the final stdout, so the cell above renders as one static block, indistinguishable from a non-streaming reply, no matter how it actually ran.
2. **Even live, Groq is probably too fast to see.** Measuring real elapsed time between chunks for a longer reply: the first ~9 chunks land within the same millisecond, and even the slower tail averages only 3-4ms between chunks — an order of magnitude faster than the human eye can register as separate frames. It's not fake, it's just fast.

`full_reply` isn't handed to you by anything, either — it only exists because *you* concatenated every piece as it arrived. That accumulation is on you, always.

To actually *watch* it happen, live, slow it down on purpose — clearly artificial, only to make the real underlying behavior visible:

In [5]:
stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
)

start = time.perf_counter()
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        time.sleep(0.15)  # artificial — not something you'd do in real code

print()
print(f"real model time was under a second; this took {time.perf_counter() - start:.1f}s on purpose")

1, 2, 3, 4, 5.
real model time was under a second; this took 2.1s on purpose


Run that cell yourself and you should genuinely see each character land, roughly a sixth of a second apart — a ~2 second demo standing in for what actually happens in well under a second.

## Events

Each `chunk` is a real object, not just a text fragment — same idea as `response` in notebook 1, but many small ones instead of one big one. The reason this matters: a chunk can carry information that isn't text at all, and the most important piece — `finish_reason` — tells your code *why* the stream ended, not just that it did. Getting that wrong means either a silently truncated reply going unnoticed, or a tool call's raw JSON getting printed to a user as if it were the answer.

First, what the pieces actually look like — the first, a middle, and the last chunk of a stream all differ:

In [ ]:
stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5."}],
    stream=True,
)
chunks = list(stream)

print("total chunks:", len(chunks))
print()
print("FIRST: ", chunks[0].choices[0].delta)
print("MIDDLE:", chunks[1].choices[0].delta)
print("LAST:  ", chunks[-1].choices[0].delta, "| finish_reason:", chunks[-1].choices[0].finish_reason)

total chunks: 16

FIRST:  ChoiceDelta(content='', function_call=None, refusal=None, role='assistant', tool_calls=None)
MIDDLE: ChoiceDelta(content=',', function_call=None, refusal=None, role=None, tool_calls=None)
LAST:   ChoiceDelta(content=None, function_call=None, refusal=None, role=None, tool_calls=None) | finish_reason: stop


The **first** chunk carries `role="assistant"` and empty content — it announces who's speaking before any text exists. **Middle** chunks carry the actual text pieces. The **last** chunk has `content=None` and is where `finish_reason` finally shows up (`"stop"` here) — you only know the reply is complete when you see this.

`finish_reason` isn't always `"stop"`. Three real scenarios, checked directly instead of assumed:

In [7]:
# 1. A reply that finishes naturally
stream = client.chat.completions.create(
    model=DEFAULT_MODEL, messages=[{"role": "user", "content": "Say hi."}], stream=True
)
print("finished naturally:", list(stream)[-1].choices[0].finish_reason)

# 2. A reply cut off by max_tokens
stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Explain quantum computing in detail."}],
    stream=True,
    max_tokens=10,
)
print("cut off by max_tokens:", list(stream)[-1].choices[0].finish_reason)

# 3. A reply that's actually a tool call
tools = [{"type": "function", "function": {
    "name": "multiply", "description": "Multiply two numbers together.",
    "parameters": {"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]},
}}]
stream = client.chat.completions.create(
    model=DEFAULT_MODEL, messages=[{"role": "user", "content": "What is 12 times 7?"}], tools=tools, stream=True
)
print("handed off to a tool:", list(stream)[-1].choices[0].finish_reason)

finished naturally: stop
cut off by max_tokens: length
handed off to a tool: tool_calls


Three different values, three different things real code has to do:

- `"stop"` — genuinely done. Display the accumulated text as the final answer.
- `"length"` — cut off by `max_tokens`, not by choice. Silently treating this like `"stop"` means shipping a truncated answer without anyone noticing; real code should flag it or continue generating.
- `"tool_calls"` — there's no final text to show at all yet. `delta.content` will have been `None` the whole time; this is the signal to switch to `execute_tool_call()`/`resolve_tools()` from notebook 6 instead of printing anything.

This is the actual payoff of understanding chunk structure: not trivia about the object's shape, but knowing which of three different code paths to take.

In [8]:
print(chunks[-1].usage)

CompletionUsage(completion_tokens=15, prompt_tokens=43, total_tokens=58, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.05197348, prompt_time=0.00123437, completion_time=0.044410039, total_time=0.045644409)


Worth knowing specifically about this stack: Groq includes real token `usage` on the final chunk automatically. OpenAI's own API doesn't do this unless you explicitly pass `stream_options={"include_usage": True}` — the cost tracking from notebook 3 works during streaming here without any extra configuration, but that's not guaranteed on every provider.

## Streaming UX

The point of streaming isn't that it's faster overall — it's that the *first* visible output arrives much sooner. That gap is real, not marketing:

In [9]:
prompt = "Write a 150 word story about a lighthouse keeper."

start = time.perf_counter()
client.chat.completions.create(model=DEFAULT_MODEL, messages=[{"role": "user", "content": prompt}])
full_latency = time.perf_counter() - start

start = time.perf_counter()
stream = client.chat.completions.create(model=DEFAULT_MODEL, messages=[{"role": "user", "content": prompt}], stream=True)
first_token_time = None
for chunk in stream:
    if chunk.choices[0].delta.content and first_token_time is None:
        first_token_time = time.perf_counter() - start
stream_total = time.perf_counter() - start

print(f"non-streaming — nothing visible until: {full_latency:.2f}s")
print(f"streaming     — first token visible at: {first_token_time:.2f}s")
print(f"streaming     — fully done at:          {stream_total:.2f}s")

non-streaming — nothing visible until: 0.95s
streaming     — first token visible at: 0.50s
streaming     — fully done at:          1.04s


The non-streaming call shows *nothing* until it's entirely finished. The streaming call shows real content roughly 5-6x sooner — that gap is what "streaming UX" actually means: perceived latency, not total latency.

## Streaming and tool calls

Tool calls can stream too. On some providers, `arguments` arrives character-by-character across many chunks, and you have to accumulate the pieces by `index` before the JSON is parseable. Worth checking directly rather than assuming:

In [10]:
tools = [
    {"type": "function", "function": {
        "name": "multiply",
        "description": "Multiply two numbers together.",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
    }},
]

stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "What is 12 times 7?"}],
    tools=tools,
    stream=True,
)

fragments = 0
for chunk in stream:
    if chunk.choices[0].delta.tool_calls:
        fragments += 1
        print(chunk.choices[0].delta.tool_calls[0].function.arguments)

print("tool-call fragments seen:", fragments)

{"a":12,"b":7}
tool-call fragments seen: 1


On Groq/Llama, the whole `arguments` string showed up in a single fragment, even for a longer, multi-field tool call tested separately. So `execute_tool_call()` from notebook 6 can be handed a streamed tool call here without extra accumulation logic. That's a property of *this* stack, not a guarantee — code intended to run against other providers (OpenAI's own models are well known for fragmenting `arguments` token-by-token) still needs to accumulate by `index` defensively rather than assume one fragment.

## From chunks to a wrapper

`stream_chat()` (in `llm.py` since notebook 1) already wraps the raw loop above into a generator of text pieces:

```python
def stream_chat(messages, model=DEFAULT_MODEL, **kwargs):
    stream = client.chat.completions.create(model=model, messages=messages, stream=True, **kwargs)
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta
```

What it doesn't give you is the full text afterward — printing every piece as it streams is nice for a UI, but you also need the complete reply to save into conversation history (`Conversation` from notebook 2 expects a plain string). `stream_and_print()` does both at once:

In [11]:
def stream_and_print(messages, model=DEFAULT_MODEL, **kwargs):
    full = []
    for delta in stream_chat(messages, model=model, **kwargs):
        print(delta, end="", flush=True)
        full.append(delta)
    print()
    return "".join(full)

reply = stream_and_print([{"role": "user", "content": "Name 3 planets."}])
print("---")
print("returned:", repr(reply))

Here are 3 planets:

1. Earth
2. Mars
3. Jupiter
---
returned: 'Here are 3 planets:\n\n1. Earth\n2. Mars\n3. Jupiter'


## What you built

✓ Proved a reply arrives in separate pieces — first as a numbered list that stays visible in saved output, then as the live-typing version with an honest caveat about what saved notebook output can and can't show

✓ Saw `finish_reason` take three different real values and what each means for what code should do next

✓ Found a Groq-specific detail: `usage` is included on the final chunk for free

✓ Measured a real ~5-6x gap between non-streaming and streaming time-to-first-visible-content

✓ Checked tool-call streaming directly rather than assuming — confirmed no fragmentation here, and why that's stack-specific, not universal

✓ Built `stream_and_print()` in `llm.py`

**Next:** `08_asyncio.ipynb` — the event loop, async/await, and running calls concurrently.